### Stream Customer data from cloud files to delta lake 
- 1.Read files using DataStreamReader API
- 2.Transfer the dataframe to add the following columns a) Filepath b)ingestion date
- 3.Write the transform data stream to delta lake

1.Read files using DataStreamReader API

In [0]:
# Step 2: now build the stream
customer_df = (spark.readStream
                    .format("cloudFiles")
                    .option("cloudFiles.format", "json")
                    .option("cloudFiles.schemaLocation", "/Volumes/gizmobox/landing/operational_data/customer_Autoloader/_schema")
                    .option("cloudFiles.inferColumnTypes", "true")
                    .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")
                    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                    .option("pathGlobFilter", "customers_2024_*.json")
                    .load("/Volumes/gizmobox/landing/operational_data/customer_Autoloader/")
)

2.Transfer the dataframe to add the following columns a) Filepath b)ingestion date

In [0]:
from pyspark.sql.functions import current_timestamp, col

customer_tranformed_df = (customer_df.withColumn("file_path",col("_metadata.file_path"))
                                     .withColumn("ingestion_date",current_timestamp())
                                     )

3.Write the transform data stream to delta lake

In [0]:
streaming_query = (customer_tranformed_df.writeStream
  .format("delta")
  .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customer_Autoloader/_checkpoints_stream")
  .option("mergeSchema", "true")
  .toTable("gizmobox.bronze.customer_Autoloader")
)


In [0]:
streaming_query.stop()

In [0]:
%sql
drop table if exists gizmobox.bronze.customer_autoloader

In [0]:
%sql
select * from gizmobox.bronze.customer_autoloader